In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DataType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

# Define schema for the data file
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
]) 

In [0]:
raw_data_path= "/Volumes/ecommerece/source_data/raw/brands/*.csv"

df= spark.read.option('header', "true").option("delimeter", ",").schema(brand_schema).csv(raw_data_path)

# add metadata columns
df= df.withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("_ingested_at", F.current_timestamp())

display(df.limit(5))

brand_code,brand_name,category_code,_source_file,_ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerece/source_data/raw/brands/brands.csv,2026-06-10T14:16:47.985Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerece/source_data/raw/brands/brands.csv,2026-06-10T14:16:47.985Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerece/source_data/raw/brands/brands.csv,2026-06-10T14:16:47.985Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerece/source_data/raw/brands/brands.csv,2026-06-10T14:16:47.985Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerece/source_data/raw/brands/brands.csv,2026-06-10T14:16:47.985Z


In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecommerece.bronze.brz_brands")

Category


In [0]:
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True),

]) 
#Load data using schema field
raw_data_path= "/Volumes/ecommerece/source_data/raw/category/*.csv"

df_raw= spark.read.option('header', "true").option("delimeter", ",").schema(category_schema).csv(raw_data_path)

# add metadata columns
df_raw= df_raw.withColumn("_ingested_at", F.current_timestamp())\
    .withColumn("_source_file", F.col("_metadata.file_path"))

# write raw data to bronze layer (catalog:ecommerece.bronze.brz_category)
df_raw.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecommerece.bronze.brz_category")

In [0]:
product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),
    StructField("length_cm", StringType(), True),
    StructField("width_cm", StringType(), True),
    StructField("height_cm", StringType(), True),
    StructField("rating_count", StringType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", StringType(), False),
]) 
                                                                                        
#Load data using schema field
raw_data_path = "/Volumes/ecommerece/source_data/raw/products/*.csv"

df= spark.read.option('header', "true").option("delimeter", ",").schema(product_schema).csv(raw_data_path)\
    .withColumn("file_name", F.col("_metadata.file_path"))\
    .withColumn("ingested_timestamp", F.current_timestamp())

# write raw data to bronze layer (catalog:ecommerece.bronze.brz_products)
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecommerece.bronze.brz_products")

In [0]:
customers_schema= StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True),
])
#Load data using schema field
raw_data_path = "/Volumes/ecommerece/source_data/raw/customers/*.csv"

df= spark.read.option('header', "true").option("delimeter", ",").schema(customers_schema).csv(raw_data_path)\
    .withColumn("file_name", F.col("_metadata.file_path"))\
    .withColumn("ingested_timestamp", F.current_timestamp())

# write raw data to bronze layer (catalog:ecommerece.bronze.brz_customers)
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecommerece.bronze.brz_customers")

In [0]:
date_schema = StructType([
    StructField("date", StringType(), True),
    StructField("year", StringType(), True),
    StructField("day_name", StringType(), True),
    StructField("quater", StringType(), True),
    StructField("week_of_year", StringType(), True),
])

#Load data using schema field
raw_data_path= "/Volumes/ecommerece/source_data/raw/date/*.csv"

df_raw= spark.read.option('header', "true").option("delimeter", ",").schema(date_schema).csv(raw_data_path) \
    .withColumn("_ingested_at", F.current_timestamp())\
    .withColumn("_source_file", F.col("_metadata.file_path"))

# write raw data to bronze layer (catalog:ecommerece.bronze.brz_calender)
df_raw.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecommerece.bronze.brz_calender")